In [6]:
"""
Beli VoC — App Store review collector (RSS endpoint).

Pulls Beli reviews from multiple Apple storefronts via Apple's public customer-reviews:
source, text, rating, version, date, title, storefront, author, review_id

Limitations:
- ~10 pages (≈500 reviews) max per storefront.
- The RSS feed carries NO review date -> 'date' is left blank.
  For clean date/version trends, lean on Google Play.
"""

import os
import time
import requests
import pandas as pd

APP_ID = 1478375386
STOREFRONTS = ["us", "gb", "ca", "au"]
MAX_PAGES = 10          # Apple caps the feed around 10 pages
SORT = "mostrecent"     
SLEEP_BETWEEN = 1       
TIMEOUT = 20
OUT_CSV = "data/app_store_reviews.csv"

HEADERS = {"User-Agent": "beli-voc-research/1.0 (review collection)"}


def _to_int(v):
    try:
        return int(v)
    except (TypeError, ValueError):
        return None


def pull_storefront(country: str) -> pd.DataFrame:
    rows = []
    for page in range(1, MAX_PAGES + 1):
        url = (f"https://itunes.apple.com/{country}/rss/customerreviews/"
               f"page={page}/id={APP_ID}/sortby={SORT}/json")
        try:
            r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            r.raise_for_status()
            entries = r.json().get("feed", {}).get("entry", [])
        except Exception as e:
            print(f"  [{country}] page {page} failed: {e}")
            break

        if not entries or len(entries) <= 1:
            break

        for e in entries:
            if "im:rating" not in e:   # skips app-metadata entry
                continue
            rows.append({
                "text": e.get("content", {}).get("label", "").strip(),
                "rating": _to_int(e.get("im:rating", {}).get("label")),
                "version": e.get("im:version", {}).get("label", ""),
                "title": e.get("title", {}).get("label", "").strip(),
                "author": e.get("author", {}).get("name", {}).get("label", ""),
            })

        time.sleep(SLEEP_BETWEEN)

    print(f"  [{country}] pulled {len(rows)} reviews")
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df["storefront"] = country
    return df


def main():
    frames = [pull_storefront(c) for c in STOREFRONTS]
    frames = [f for f in frames if not f.empty]
    if not frames:
        print("No reviews pulled.")
        return

    raw = pd.concat(frames, ignore_index=True)

    out = pd.DataFrame({
        "source": "app_store",
        "text": raw["text"].astype(str).str.strip(),
        "rating": raw["rating"],
        "version": raw["version"].astype(str).str.strip(),
        "date": pd.NaT,  # RSS feed provides no review date
        "title": raw["title"].astype(str).str.strip(),
        "storefront": raw["storefront"],
        "author": raw["author"].astype(str).str.strip(),
    })

    out = out[out["text"].str.len() > 0]
    out = out.drop_duplicates(subset=["text", "author", "storefront"]).reset_index(drop=True)
    out["review_id"] = out.index

    os.makedirs("data", exist_ok=True)
    out.to_csv(OUT_CSV, index=False)

    print(f"\nWrote {len(out)} unique reviews -> {OUT_CSV}")
    print("By storefront:\n", out["storefront"].value_counts().to_string())
    print("By rating:\n", out["rating"].value_counts().sort_index().to_string())
    print("Top versions:\n", out["version"].value_counts().head(10).to_string())


if __name__ == "__main__":
    main()

  [us] pulled 500 reviews
  [gb] pulled 17 reviews
  [ca] pulled 41 reviews
  [au] pulled 4 reviews

Wrote 562 unique reviews -> data/app_store_reviews.csv
By storefront:
 storefront
us    500
ca     41
gb     17
au      4
By rating:
 rating
1    146
2     30
3     63
4     92
5    231
Top versions:
 version
7.2.1    12
7.5.1    11
6.3.0    11
8.1.6    10
7.7.4    10
3.7.0    10
7.2.7     8
9.0.4     8
8.2.1     8
7.6.8     7


In [8]:
"""
Beli VoC — Google Play review collector.

Pulls Beli reviews from Google Play via google-play-scraper, dedupes, and writes a
tidy CSV with the SAME schema as the App Store collector:
    source, text, rating, version, date, title, storefront, author, review_id

Why this source matters: unlike Apple, Google Play attaches BOTH an app version
and a real date to every review -> this is the backbone of the across-versions /
over-time trend analysis (RQ4).

Set PACKAGE_NAME from the Play listing URL: play.google.com/...?id=<PACKAGE_NAME>
"""

import os
import pandas as pd
from google_play_scraper import reviews, Sort

PACKAGE_NAME = "com.beliapp.myapp"   
LANG = "en"
COUNTRY = "us"
HOW_MANY = 2000       
OUT_CSV = "data/google_play_reviews.csv"


def main():
    print(f"Pulling up to {HOW_MANY} reviews for {PACKAGE_NAME} ...")
    result, _ = reviews(
        PACKAGE_NAME,
        lang=LANG,
        country=COUNTRY,
        sort=Sort.NEWEST,     
        count=HOW_MANY,
    )
    print(f"  pulled {len(result)} raw reviews")

    if not result:
        print("No reviews pulled. Double-check PACKAGE_NAME against the Play URL.")
        return

    raw = pd.DataFrame(result)

    out = pd.DataFrame({
        "source": "google_play",
        "text": raw["content"].astype(str).str.strip(),
        "rating": raw["score"],
        "version": raw.get("reviewCreatedVersion", "").astype(str).str.strip(),
        "date": pd.to_datetime(raw["at"], errors="coerce"),
        "title": "",  # Play reviews have no title field
        "storefront": COUNTRY,
        "author": raw["userName"].astype(str).str.strip(),
    })

    out = out[out["text"].str.len() > 0]
    out = out.drop_duplicates(subset=["text", "author", "date"]).reset_index(drop=True)
    out["review_id"] = out.index

    os.makedirs("data", exist_ok=True)
    out.to_csv(OUT_CSV, index=False)

    print(f"\nWrote {len(out)} unique reviews -> {OUT_CSV}")
    print(f"Date range: {out['date'].min()} -> {out['date'].max()}")
    print("By rating:\n", out["rating"].value_counts().sort_index().to_string())
    print("Reviews with a version tag:", (out["version"].str.len() > 0).sum())
    print("Top versions:\n", out["version"].value_counts().head(10).to_string())


if __name__ == "__main__":
    main()

Pulling up to 2000 reviews for com.beliapp.myapp ...
  pulled 263 raw reviews

Wrote 263 unique reviews -> data/google_play_reviews.csv
Date range: 2022-10-19 03:26:30 -> 2026-06-21 09:32:35
By rating:
 rating
1    76
2    34
3    28
4    36
5    89
Reviews with a version tag: 263
Top versions:
 version
None      41
6.3.0     11
8.2.31     6
8.4.12     6
7.6.11     6
7.9.6      6
7.2.2      5
6.2.1      5
7.7.4      5
7.8.2      5
